# 💊 Pharmaceutical Sales Forecasting
## From Historical Patterns to Forward-Looking Demand Signals

**Author:** Sarat Bhattiprolu  
**Dataset:** [Pharmaceutical Drug Spending Dataset — Kaggle](https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data)  
**Tools:** Python · ARIMA · Facebook Prophet · statsmodels

---

### What This Notebook Does

This notebook takes historical weekly pharmaceutical sales data across **8 drug categories** and produces two operational forecasting outputs:

| Forecast Horizon | Model | Business Use |
|---|---|---|
| **12 weeks ahead** | ARIMA (per drug) | Weekly replenishment & S&OP planning |
| **52 weeks ahead** | Prophet (per drug) | Annual demand planning & procurement |

Each drug category is modelled separately using the best-fit model identified through prior evaluation. Forecasts include **confidence/uncertainty intervals** so planners can make risk-informed decisions, not just point estimates.

### Drug Categories Covered

| ATC Code | Drug | Category | Predictability Tier |
|---|---|---|---|
| M01AB | Diclofenac | Anti-inflammatory | 🟢 Tier 1 — Highly Predictable |
| M01AE | Ibuprofen | Anti-inflammatory | 🟢 Tier 1 — Highly Predictable |
| N02BA | Aspirin | Analgesic | 🟡 Tier 1 — Watch: Structural Decline |
| N02BE | Paracetamol | Analgesic | 🟢 Tier 1 — Highly Predictable |
| N05B | Anxiolytics | CNS / Psych | 🟡 Tier 2 — Moderate Noise |
| N05C | Sedatives | CNS / Psych | 🔴 Tier 3 — High Volatility |
| R03 | Respiratory Drugs | Respiratory | 🟡 Tier 2 — Strongly Seasonal |
| R06 | Antihistamines | Allergy | 🟡 Tier 2 — Strongly Seasonal |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# ── Output folders ─────────────────────────────────────────
Path('outputs/forecasts').mkdir(parents=True, exist_ok=True)
Path('outputs/charts').mkdir(parents=True, exist_ok=True)

# ── Visual style ───────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

PALETTE = {
    'actual':     '#1E3A5F',
    'arima':      '#0F7EB8',
    'prophet':    '#028090',
    'ci_arima':   '#B5D4F4',
    'ci_prophet': '#9FE1CB',
    'tier1':      '#028090',
    'tier2':      '#D97706',
    'tier3':      '#C0392B',
    'aspirin':    '#F59E0B',
}

print("✅ Imports and config loaded.")


In [ ]:
# ── Drug-level configuration ──────────────────────────────
# ARIMA orders: from rolling forecast evaluation (best MSE per drug)
# Prophet CPS:  from grid-search — M01AB corrected from 30→0.3 (30 caused
#               overfitting to historical noise, flattening the forecast)
# All drugs use logistic growth with floor=0 (sales cannot be negative)

import pandas as pd
import numpy as np

DRUG_CONFIG = {
    'M01AB': {
        'name':    'Diclofenac',
        'label':   'M01AB · Diclofenac (Anti-inflammatory)',
        'arima':   (0, 0, 0),
        'prophet': {'changepoint_prior_scale': 0.3},   # corrected from 30
        'seasonal': False,
        'tier':    1,
        'tier_color': PALETTE['tier1'],
        'noise_pct': 14.8,
    },
    'M01AE': {
        'name':    'Ibuprofen',
        'label':   'M01AE · Ibuprofen (Anti-inflammatory)',
        'arima':   (2, 0, 0),
        'prophet': {'changepoint_prior_scale': 0.05},
        'seasonal': False,
        'tier':    1,
        'tier_color': PALETTE['tier1'],
        'noise_pct': 15.4,
    },
    'N02BA': {
        'name':    'Aspirin',
        'label':   'N02BA · Aspirin (Analgesic) ⚠ Structural Decline',
        'arima':   (5, 1, 1),
        'prophet': {'changepoint_prior_scale': 0.05},
        'seasonal': False,
        'tier':    1,
        'tier_color': PALETTE['aspirin'],
        'noise_pct': 14.5,
        'flag':    'structural_decline',
    },
    'N02BE': {
        'name':    'Paracetamol',
        'label':   'N02BE · Paracetamol (Analgesic)',
        'arima':   (2, 0, 0),
        'prophet': {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 170},
        'seasonal': True,
        'tier':    1,
        'tier_color': PALETTE['tier1'],
        'noise_pct': 13.6,
    },
    'N05B': {
        'name':    'Anxiolytics',
        'label':   'N05B · Anxiolytics (CNS)',
        'arima':   (0, 0, 5),
        'prophet': {'changepoint_prior_scale': 0.3},
        'seasonal': False,
        'tier':    2,
        'tier_color': PALETTE['tier2'],
        'noise_pct': 20.9,
    },
    'N05C': {
        'name':    'Sedatives',
        'label':   'N05C · Sedatives (CNS) ⚠ High Volatility',
        'arima':   (0, 0, 1),
        'prophet': {'changepoint_prior_scale': 0.5},
        'seasonal': False,
        'tier':    3,
        'tier_color': PALETTE['tier3'],
        'noise_pct': 52.6,
        'flag':    'high_volatility',
    },
    'R03': {
        'name':    'Respiratory',
        'label':   'R03 · Respiratory Drugs',
        'arima':   (5, 1, 1),
        'prophet': {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 160},
        'seasonal': True,
        'tier':    2,
        'tier_color': PALETTE['tier2'],
        'noise_pct': 29.3,
    },
    'R06': {
        'name':    'Antihistamines',
        'label':   'R06 · Antihistamines (Allergy)',
        'arima':   (1, 0, 1),
        'prophet': {'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 120},
        'seasonal': True,
        'tier':    2,
        'tier_color': PALETTE['tier2'],
        'noise_pct': 21.7,
    },
}

DRUGS         = list(DRUG_CONFIG.keys())
HORIZON_SHORT = 12   # weeks — ARIMA
HORIZON_LONG  = 52   # weeks — Prophet

print(f"✅ Config ready: {len(DRUGS)} drugs, {HORIZON_SHORT}w short-term, {HORIZON_LONG}w long-term.")


---
## Section 1 · Data Loading & Historical Overview

We load the weekly sales data and visualise the complete historical record for all eight drug categories. This gives us the baseline picture before any forecasting begins.

> **Data note:** Each value represents weekly dispensing volume (units sold) at the pharmacy level. The dataset spans approximately 6 years of weekly observations.


In [ ]:
df = pd.read_csv('salesweekly.csv', parse_dates=['datum'])
df = df.sort_values('datum').reset_index(drop=True)

print(f"Dataset: {df.shape[0]} weekly observations")
print(f"Period:  {df['datum'].min().date()} → {df['datum'].max().date()}")
print(f"Drugs:   {DRUGS}")
print()
print(df[DRUGS].describe().round(1))


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle('Historical Weekly Sales — All Drug Categories\n(Full Dataset Used for Forecasting)',
             fontsize=16, fontweight='bold', y=1.01)

for ax, drug in zip(axes.flat, DRUGS):
    cfg = DRUG_CONFIG[drug]
    color = cfg['tier_color']

    # 4-week rolling mean for clarity
    raw  = df[drug]
    roll = raw.rolling(4, center=True).mean()

    ax.fill_between(df['datum'], raw, alpha=0.15, color=color)
    ax.plot(df['datum'], raw,  color=color, linewidth=0.6, alpha=0.5, label='Weekly')
    ax.plot(df['datum'], roll, color=color, linewidth=2.0,             label='4-wk avg')

    ax.set_title(cfg['label'], fontweight='bold', pad=8)
    ax.set_ylabel('Units sold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

    # Tier badge
    badge_color = cfg['tier_color']
    ax.text(0.02, 0.94, f"Tier {cfg['tier']} · {cfg['noise_pct']}% noise",
            transform=ax.transAxes, fontsize=9,
            color='white', fontweight='bold',
            bbox=dict(facecolor=badge_color, edgecolor='none', pad=3, boxstyle='round,pad=0.3'))

    ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('outputs/charts/00_historical_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Saved: outputs/charts/00_historical_overview.png")


---
## Section 2 · Short-Term Forecasting — ARIMA (12 Weeks Ahead)

### Why ARIMA for short-term?

ARIMA (AutoRegressive Integrated Moving Average) was validated as the best-performing model for short-term rolling forecasts across 7 of 8 drug categories in the evaluation phase. It excels at capturing recent autocorrelation patterns and produces well-calibrated confidence intervals.

**What we do here:**
- Train each drug's ARIMA model on the **complete historical dataset** (not a train/test split — this is real forecasting)
- Generate **12 weeks of forward predictions** from the last observed date
- Produce **95% confidence intervals** for risk-informed planning

**Operational use:** These 12-week forecasts directly feed weekly S&OP replenishment decisions. The confidence interval tells your supply chain team the plausible demand range — not just a single number.

| Drug | ARIMA Order | Notes |
|---|---|---|
| M01AB | (0,0,0) | White noise — forecast = mean |
| M01AE | (2,0,0) | AR(2) — recent weeks inform next |
| N02BA | (5,1,1) | Differenced — structural trend |
| N02BE | (2,0,0) | AR(2) |
| N05B  | (0,0,5) | MA(5) — moving average errors |
| N05C  | (0,0,1) | MA(1) — high residual noise |
| R03   | (5,1,1) | Differenced — seasonal trend |
| R06   | (1,0,1) | ARMA(1,1) |


In [ ]:
def arima_forecast(series, order, horizon, alpha=0.05):
    """
    Fit ARIMA on full series, return horizon-step forecast with CIs.

    Parameters
    ----------
    series  : pd.Series  — full historical time series
    order   : tuple      — (p, d, q)
    horizon : int        — weeks ahead to forecast
    alpha   : float      — CI width (0.05 = 95%)

    Returns
    -------
    dict with keys: mean, lower, upper (all pd.Series indexed by future dates)
    """
    model  = ARIMA(series.values, order=order)
    fitted = model.fit()

    fc     = fitted.get_forecast(steps=horizon)
    mean   = fc.predicted_mean
    ci     = fc.conf_int(alpha=alpha)

    # Build future date index (weekly frequency)
    last_date    = series.index[-1] if hasattr(series.index, 'freq') else series.values[-1]
    future_dates = pd.date_range(start=last_date, periods=horizon + 1, freq='W')[1:]

    return {
        'mean':  pd.Series(mean,       index=future_dates, name='forecast'),
        'lower': pd.Series(ci[:, 0],   index=future_dates, name='lower_95'),
        'upper': pd.Series(ci[:, 1],   index=future_dates, name='upper_95'),
        'aic':   round(fitted.aic, 2),
        'order': order,
    }

print("✅ ARIMA forecast function defined.")


In [ ]:
arima_results = {}

for drug, cfg in DRUG_CONFIG.items():
    series = df.set_index('datum')[drug]
    order  = cfg['arima']

    print(f"  Fitting ARIMA{order} on {drug} ({len(series)} obs)...", end=' ')
    result = arima_forecast(series, order, HORIZON_SHORT)
    arima_results[drug] = result
    print(f"AIC={result['aic']}")

print(f"\n✅ ARIMA forecasts complete for all {len(DRUGS)} drugs.")
print(f"   Horizon: {HORIZON_SHORT} weeks")
print(f"   Forecast period: {list(arima_results.values())[0]['mean'].index[0].date()} → "
      f"{list(arima_results.values())[0]['mean'].index[-1].date()}")


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle(f'ARIMA Short-Term Forecasts — {HORIZON_SHORT} Weeks Ahead\n'
             'Shaded band = 95% confidence interval',
             fontsize=16, fontweight='bold', y=1.01)

# Show only last 52 weeks of history for readability
lookback = 52

for ax, drug in zip(axes.flat, DRUGS):
    cfg  = DRUG_CONFIG[drug]
    fc   = arima_results[drug]
    color = PALETTE['arima']
    tc    = cfg['tier_color']

    series = df.set_index('datum')[drug]
    hist   = series.iloc[-lookback:]

    # History
    ax.plot(hist.index, hist.values,
            color=PALETTE['actual'], linewidth=1.5, label='Historical (last 52w)')

    # Forecast + CI
    ax.fill_between(fc['mean'].index, fc['lower'], fc['upper'],
                    color=PALETTE['ci_arima'], alpha=0.5, label='95% CI')
    ax.plot(fc['mean'].index, fc['mean'],
            color=color, linewidth=2.5, marker='o', markersize=3,
            label=f"ARIMA{cfg['arima']} forecast")

    # Vertical divider
    ax.axvline(x=series.index[-1], color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(series.index[-1], ax.get_ylim()[1] * 0.95, ' Forecast →',
            fontsize=7, color='gray', va='top')

    ax.set_title(cfg['label'], fontweight='bold', pad=8)
    ax.set_ylabel('Units sold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    # Tier badge
    ax.text(0.02, 0.94, f"Tier {cfg['tier']} · {cfg['noise_pct']}% noise",
            transform=ax.transAxes, fontsize=8, color='white', fontweight='bold',
            bbox=dict(facecolor=tc, edgecolor='none', boxstyle='round,pad=0.3'))

    ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('outputs/charts/01_arima_forecasts.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Saved: outputs/charts/01_arima_forecasts.png")


---
## Section 3 · Long-Term Forecasting — Prophet (52 Weeks Ahead)

### Why Prophet for long-term?

Facebook Prophet handles seasonality, trend changepoints, and missing data more gracefully than ARIMA at extended horizons. In evaluation, Prophet significantly outperformed ARIMA for long-term forecasts on seasonal drug categories (R03, R06) and produced competitive results across all others.

**What we do here:**
- Train Prophet on the **complete historical dataset**
- Generate **52 weeks of forward predictions** — a full annual planning horizon
- Produce **uncertainty intervals** that widen appropriately over time (unlike ARIMA which can produce unrealistically tight long-range CIs)

**Operational use:** These annual forecasts feed procurement strategy, supplier contracting, and budget planning. For seasonal drugs (R03, R06), the forecast explicitly models the expected demand peaks — giving supply chain teams actionable lead time.

**Key hyperparameters used** (validated via grid-search in analysis phase):

| Drug | changepoint_prior_scale | seasonality_prior_scale | Seasonal model |
|---|---|---|---|
| M01AB | 30.0 | default | None |
| M01AE | 0.05 | default | None |
| N02BA | 0.01 | default | None |
| N02BE | 10.0 | 170 | Yearly |
| N05B  | 5.0  | default | None |
| N05C  | 0.5  | default | None |
| R03   | 0.05 | 160 | Yearly |
| R06   | 0.05 | 120 | Yearly |


In [ ]:
def prophet_forecast(series_df, cfg, horizon):
    """
    Fit Prophet on full series using logistic growth (floor=0).
    Returns horizon-week forecast with uncertainty bands.

    Logistic growth is used for all drugs because:
    - Sales cannot be negative (floor=0 enforces this)
    - Demand cannot grow unboundedly (cap = 2x historical max)
    - For stable/declining drugs this produces more realistic CIs
      than linear growth which can extrapolate below zero

    Parameters
    ----------
    series_df : pd.DataFrame  — columns ['ds', 'y']
    cfg       : dict          — drug config (prophet params, seasonal flag)
    horizon   : int           — weeks ahead

    Returns
    -------
    dict with keys: forecast, future, model
    """
    params = cfg['prophet']

    # Set cap and floor — required for logistic growth
    hist_max = series_df['y'].max()
    cap      = hist_max * 2.0    # headroom above historical peak
    floor    = 0.0               # sales cannot go negative

    df_fit = series_df.copy()
    df_fit['cap']   = cap
    df_fit['floor'] = floor

    model = Prophet(
        growth                  = 'logistic',
        changepoint_prior_scale = params.get('changepoint_prior_scale', 0.05),
        interval_width          = 0.95,
        daily_seasonality       = False,
        weekly_seasonality      = False,
        yearly_seasonality      = False,
    )

    if cfg.get('seasonal', False):
        sps = params.get('seasonality_prior_scale', 10)
        model.add_seasonality(name='yearly', period=365.25,
                              fourier_order=10, prior_scale=sps)

    model.fit(df_fit, iter=1000)

    future          = model.make_future_dataframe(periods=horizon, freq='W')
    future['cap']   = cap
    future['floor'] = floor

    forecast = model.predict(future)

    # Clip to physical bounds (logistic should already enforce this,
    # but numerical edge cases can still produce tiny negatives)
    for col in ['yhat', 'yhat_lower', 'yhat_upper']:
        forecast[col] = forecast[col].clip(lower=0)

    last_obs  = series_df['ds'].max()
    fc_future = forecast[forecast['ds'] > last_obs].copy()

    return {'forecast': forecast, 'future': fc_future, 'model': model,
            'cap': cap, 'floor': floor}

print("✅ Prophet forecast function defined (logistic growth, floor=0).")


In [ ]:
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

prophet_results = {}

for drug, cfg in DRUG_CONFIG.items():
    series_df = df[['datum', drug]].rename(columns={'datum': 'ds', drug: 'y'})
    seasonal_tag = '(seasonal)' if cfg['seasonal'] else ''

    print(f"  Fitting Prophet on {drug} {seasonal_tag}...", end=' ', flush=True)
    result = prophet_forecast(series_df, cfg, HORIZON_LONG)
    prophet_results[drug] = result
    fc = result['future']
    print(f"forecast range [{fc['yhat'].min():.1f}, {fc['yhat'].max():.1f}]")

print(f"\n✅ Prophet forecasts complete for all {len(DRUGS)} drugs.")
print(f"   Horizon: {HORIZON_LONG} weeks")
last_drug_fc = list(prophet_results.values())[-1]['future']
print(f"   Forecast period: {last_drug_fc['ds'].iloc[0].date()} → {last_drug_fc['ds'].iloc[-1].date()}")


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle(
    f'Prophet Long-Term Forecasts — {HORIZON_LONG} Weeks Ahead\n'
    'Logistic growth · floor=0 · shaded = 95% uncertainty interval',
    fontsize=16, fontweight='bold', y=1.01
)

for ax, drug in zip(axes.flat, DRUGS):
    cfg    = DRUG_CONFIG[drug]
    res    = prophet_results[drug]
    fc_fut = res['future']
    tc     = cfg['tier_color']

    series    = df.set_index('datum')[drug]
    last_date = series.index[-1]

    # ── Full historical actuals ──────────────────────────────
    # Show everything — thin for early years, thicker for recent
    cutoff_recent = series.index[-52]
    ax.plot(series.index, series.values,
            color=PALETTE['actual'], linewidth=0.8, alpha=0.35, label='_nolegend_')
    ax.plot(series.loc[cutoff_recent:].index,
            series.loc[cutoff_recent:].values,
            color=PALETTE['actual'], linewidth=1.8, label='Historical (last 52w)')

    # ── Future forecast + uncertainty band ───────────────────
    ax.fill_between(fc_fut['ds'], fc_fut['yhat_lower'], fc_fut['yhat_upper'],
                    color=PALETTE['ci_prophet'], alpha=0.5, label='95% uncertainty')
    ax.plot(fc_fut['ds'], fc_fut['yhat'],
            color=PALETTE['prophet'], linewidth=2.5, label='Prophet forecast')

    # ── Forecast cutoff line ─────────────────────────────────
    ax.axvline(x=last_date, color='#444', linestyle='--', linewidth=1.2, alpha=0.7)
    ymax = ax.get_ylim()[1]
    ax.text(last_date, ymax * 0.97, ' Forecast →',
            fontsize=8, color='#444', va='top', fontstyle='italic')

    # ── Ensure y-axis starts at 0 (floor enforced) ───────────
    ax.set_ylim(bottom=0)

    # ── Annotation for non-seasonal stable drugs ─────────────
    if not cfg['seasonal'] and cfg.get('flag') != 'structural_decline':
        ax.text(0.98, 0.06,
                'Stable demand — flat forecast is correct\nSee ARIMA for week-by-week signal',
                transform=ax.transAxes, fontsize=7.5, color='#555',
                ha='right', va='bottom',
                bbox=dict(facecolor='white', edgecolor='#ccc', boxstyle='round,pad=0.3'))

    if cfg.get('flag') == 'structural_decline':
        ax.text(0.98, 0.06,
                '⚠ Declining trend confirmed by model',
                transform=ax.transAxes, fontsize=7.5, color=PALETTE['aspirin'],
                ha='right', va='bottom', fontweight='bold',
                bbox=dict(facecolor='#FEF3C7', edgecolor=PALETTE['aspirin'], boxstyle='round,pad=0.3'))

    if cfg.get('flag') == 'high_volatility':
        ax.text(0.98, 0.06,
                '⚠ Wide CI reflects genuine volatility — plan to upper bound',
                transform=ax.transAxes, fontsize=7.5, color=PALETTE['tier3'],
                ha='right', va='bottom', fontweight='bold',
                bbox=dict(facecolor='#FEF2F2', edgecolor=PALETTE['tier3'], boxstyle='round,pad=0.3'))

    ax.set_title(cfg['label'], fontweight='bold', pad=8)
    ax.set_ylabel('Units sold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    # Tier badge
    ax.text(0.02, 0.94, f"Tier {cfg['tier']} · {cfg['noise_pct']}% noise",
            transform=ax.transAxes, fontsize=8, color='white', fontweight='bold',
            bbox=dict(facecolor=tc, edgecolor='none', boxstyle='round,pad=0.3'))

    ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('outputs/charts/02_prophet_forecasts.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Saved: outputs/charts/02_prophet_forecasts.png")


---
## Section 4 · Forecast Summary Dashboard

A single-view comparison of both forecast horizons across all drug categories, alongside their predictability tier — giving decision-makers a portfolio-level picture of expected demand.


In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.suptitle('Pharma Demand Forecast — Portfolio Summary Dashboard',
             fontsize=17, fontweight='bold', y=1.01)

# ── Left panel: 12-week ARIMA summary bars ──────────────────
ax_left = fig.add_subplot(1, 3, (1, 2))
ax_left.set_facecolor('#F8F9FA')

drug_labels, mean_12w, lower_12w, upper_12w, colors_12w = [], [], [], [], []

for drug, cfg in DRUG_CONFIG.items():
    fc = arima_results[drug]
    drug_labels.append(cfg['name'])
    mean_12w.append(fc['mean'].mean())
    lower_12w.append(fc['lower'].mean())
    upper_12w.append(fc['upper'].mean())
    colors_12w.append(cfg['tier_color'])

y_pos = np.arange(len(drug_labels))
bars  = ax_left.barh(y_pos, mean_12w, color=colors_12w, alpha=0.85, height=0.55)

# CI error bars
xerr_lo = [m - l for m, l in zip(mean_12w, lower_12w)]
xerr_hi = [u - m for m, u in zip(mean_12w, upper_12w)]
ax_left.errorbar(mean_12w, y_pos, xerr=[xerr_lo, xerr_hi],
                 fmt='none', color='#333', capsize=4, linewidth=1.5)

ax_left.set_yticks(y_pos)
ax_left.set_yticklabels(drug_labels, fontsize=11)
ax_left.set_xlabel('Average weekly units (forecast period)', fontsize=11)
ax_left.set_title(f'12-Week ARIMA Forecast — Mean Demand with 95% CI', fontweight='bold')
ax_left.grid(axis='x', alpha=0.4)

# Value labels
for i, (bar, val) in enumerate(zip(bars, mean_12w)):
    ax_left.text(val + max(mean_12w)*0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

# Tier legend
tier_patches = [
    mpatches.Patch(color=PALETTE['tier1'],  label='Tier 1 — Highly Predictable'),
    mpatches.Patch(color=PALETTE['tier2'],  label='Tier 2 — Moderate Noise'),
    mpatches.Patch(color=PALETTE['tier3'],  label='Tier 3 — High Volatility'),
    mpatches.Patch(color=PALETTE['aspirin'],label='Tier 1 — Watch: Structural Trend'),
]
ax_left.legend(handles=tier_patches, loc='lower right', fontsize=8)

# ── Right panel: noise vs forecast range scatter ─────────────
ax_right = fig.add_subplot(1, 3, 3)
ax_right.set_facecolor('#F8F9FA')

for drug, cfg in DRUG_CONFIG.items():
    fc_p   = prophet_results[drug]['future']
    spread = (fc_p['yhat_upper'] - fc_p['yhat_lower']).mean()
    noise  = cfg['noise_pct']
    color  = cfg['tier_color']

    ax_right.scatter(noise, spread, s=120, color=color, zorder=5, edgecolors='white', linewidth=1.5)
    ax_right.annotate(cfg['name'], (noise, spread),
                      textcoords='offset points', xytext=(6, 3), fontsize=8, color='#333')

ax_right.set_xlabel('Historical demand noise (%)', fontsize=11)
ax_right.set_ylabel('Avg forecast uncertainty band (units)', fontsize=11)
ax_right.set_title('Noise vs Forecast Uncertainty\n(Prophet 52-week)', fontweight='bold')
ax_right.grid(alpha=0.4)

plt.tight_layout()
plt.savefig('outputs/charts/03_summary_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Saved: outputs/charts/03_summary_dashboard.png")


---
## Section 5 · Export Forecast Outputs

All forecasts are saved to `outputs/forecasts/` as CSVs — ready to load into any planning system, Excel model, or BI tool.

**Files generated:**
- `arima_12w_forecasts.csv` — all drugs, 12-week ARIMA forecasts with CIs
- `prophet_52w_forecasts.csv` — all drugs, 52-week Prophet forecasts with uncertainty bands
- `forecast_summary.csv` — one row per drug: key stats for both horizons


In [ ]:
# ── ARIMA export ───────────────────────────────────────────
arima_rows = []
for drug, cfg in DRUG_CONFIG.items():
    fc = arima_results[drug]
    for date, mean, lower, upper in zip(fc['mean'].index, fc['mean'], fc['lower'], fc['upper']):
        arima_rows.append({
            'drug_code':    drug,
            'drug_name':    cfg['name'],
            'tier':         cfg['tier'],
            'noise_pct':    cfg['noise_pct'],
            'forecast_date': date.date(),
            'forecast_mean': round(mean, 2),
            'lower_95':      round(lower, 2),
            'upper_95':      round(upper, 2),
            'model':        f'ARIMA{cfg["arima"]}',
            'horizon':      'short_term_12w',
        })

arima_df = pd.DataFrame(arima_rows)
arima_df.to_csv('outputs/forecasts/arima_12w_forecasts.csv', index=False)
print(f"✅ Saved ARIMA forecasts: {len(arima_df)} rows")
print(arima_df.head(4).to_string())

print()

# ── Prophet export ──────────────────────────────────────────
prophet_rows = []
for drug, cfg in DRUG_CONFIG.items():
    fc = prophet_results[drug]['future']
    for _, row in fc.iterrows():
        prophet_rows.append({
            'drug_code':    drug,
            'drug_name':    cfg['name'],
            'tier':         cfg['tier'],
            'noise_pct':    cfg['noise_pct'],
            'forecast_date': row['ds'].date(),
            'forecast_mean': round(row['yhat'], 2),
            'lower_95':      round(row['yhat_lower'], 2),
            'upper_95':      round(row['yhat_upper'], 2),
            'model':        'Prophet',
            'horizon':      'long_term_52w',
        })

prophet_df = pd.DataFrame(prophet_rows)
prophet_df.to_csv('outputs/forecasts/prophet_52w_forecasts.csv', index=False)
print(f"✅ Saved Prophet forecasts: {len(prophet_df)} rows")
print(prophet_df.head(4).to_string())


In [ ]:
# ── Combined summary stats ─────────────────────────────────
summary_rows = []
for drug, cfg in DRUG_CONFIG.items():
    afc = arima_results[drug]
    pfc = prophet_results[drug]['future']

    summary_rows.append({
        'drug_code':           drug,
        'drug_name':           cfg['name'],
        'tier':                cfg['tier'],
        'historical_noise_pct': cfg['noise_pct'],
        'arima_order':         str(cfg['arima']),
        'arima_12w_mean':      round(afc['mean'].mean(), 1),
        'arima_12w_lower':     round(afc['lower'].mean(), 1),
        'arima_12w_upper':     round(afc['upper'].mean(), 1),
        'arima_12w_ci_width':  round((afc['upper'] - afc['lower']).mean(), 1),
        'prophet_52w_mean':    round(pfc['yhat'].mean(), 1),
        'prophet_52w_lower':   round(pfc['yhat_lower'].mean(), 1),
        'prophet_52w_upper':   round(pfc['yhat_upper'].mean(), 1),
        'prophet_52w_ci_width': round((pfc['yhat_upper'] - pfc['yhat_lower']).mean(), 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('outputs/forecasts/forecast_summary.csv', index=False)

print("✅ Forecast Summary Table")
print("=" * 90)
display_cols = ['drug_code','drug_name','tier','historical_noise_pct',
                'arima_12w_mean','arima_12w_ci_width',
                'prophet_52w_mean','prophet_52w_ci_width']
print(summary_df[display_cols].to_string(index=False))
print()
print("✅ Saved: outputs/forecasts/forecast_summary.csv")


---
## Section 6 · Business Interpretation

### What to do with these forecasts

#### Tier 1 — Analgesics & Anti-inflammatories (M01AB, M01AE, N02BA, N02BE)

These drugs are your most forecastable. The ARIMA 12-week signal is reliable enough to drive **automated replenishment decisions** with minimal human override. Target safety stock should reflect the 95% CI upper bound — not the mean — to maintain service levels without significant overstock.

> **N02BA (Aspirin) caveat:** the 5th-order differenced ARIMA model is accounting for the structural downward trend. Monitor whether the actual demand tracks the lower end of the CI over the coming quarters — that would confirm the structural decline hypothesis.

#### Tier 2 — Seasonal Categories (R03, R06)

The Prophet 52-week forecast is the critical planning tool here. The model explicitly captures the seasonal demand wave — **do not use the flat ARIMA mean for annual procurement decisions on these drugs.** The peak-to-trough difference in the Prophet forecast is your procurement swing that needs to be pre-positioned 6–8 weeks in advance.

#### Tier 2 — Anxiolytics (N05B)

Moderate noise. Use the ARIMA 12-week for replenishment but apply a conservative buffer on top of the CI upper bound (~10–15% safety stock premium).

#### Tier 3 — Sedatives (N05C)

The forecast CI is wide by design — this reflects genuine demand uncertainty, not poor modelling. **Do not narrow this CI by using a shorter lookback window.** The wide band is the correct signal. Set safety stock to the 95% upper bound and maintain a manual review cadence for any external signals (regulatory changes, prescribing guideline updates) that the model cannot anticipate.

---

### What this project demonstrates

| Skill | Evidence |
|---|---|
| Time-series forecasting (ARIMA, Prophet) | Per-drug model selection based on evaluation results |
| Uncertainty quantification | 95% CI on every forecast — not just point estimates |
| Business-driven model selection | Different models for different operational horizons |
| Production-ready outputs | Structured CSV exports ready for planning system integration |
| Portfolio-quality communication | Forecast outputs tied to specific supply chain decisions |

---

*Data source: Pharmaceutical Drug Spending Dataset, Kaggle.  
Models: statsmodels ARIMA, Facebook Prophet.  
This notebook is part of a data science portfolio project.*


In [ ]:
readme = """
# 💊 Pharma Sales Forecasting

> Turning historical pharmaceutical sales data into actionable forward-looking demand signals using ARIMA and Facebook Prophet.

## Business Problem

Pharmaceutical supply chains need accurate demand signals at two horizons:
- **12 weeks ahead** for weekly replenishment and S&OP planning
- **52 weeks ahead** for annual procurement and supplier contracting

This project builds those signals for 8 drug categories across the ATC classification system.

## Approach

| Stage | What was done |
|---|---|
| Exploratory Analysis | Stationarity tests, seasonality decomposition, noise profiling |
| Model Evaluation | ARIMA, Auto-ARIMA, Prophet, Vanilla/Stacked/Bidirectional LSTM |
| Forecasting | Best model per drug trained on full data → real future forecasts |
| Output | CSVs with point forecasts + 95% confidence/uncertainty intervals |

## Drug Portfolio

| Code | Drug | Tier | Noise |
|---|---|---|---|
| M01AB | Diclofenac | Tier 1 — Highly Predictable | 14.8% |
| M01AE | Ibuprofen | Tier 1 — Highly Predictable | 15.4% |
| N02BA | Aspirin | Tier 1 — Structural Decline Watch | 14.5% |
| N02BE | Paracetamol | Tier 1 — Highly Predictable | 13.6% |
| N05B | Anxiolytics | Tier 2 — Moderate Noise | 20.9% |
| N05C | Sedatives | Tier 3 — High Volatility | 52.6% |
| R03 | Respiratory Drugs | Tier 2 — Strongly Seasonal | 29.3% |
| R06 | Antihistamines | Tier 2 — Strongly Seasonal | 21.7% |

## Outputs

- `outputs/forecasts/arima_12w_forecasts.csv` — 12-week forward forecasts (all drugs)
- `outputs/forecasts/prophet_52w_forecasts.csv` — 52-week forward forecasts (all drugs)
- `outputs/forecasts/forecast_summary.csv` — Portfolio-level summary table
- `outputs/charts/` — Publication-quality forecast visualisations

## Setup

```bash
pip install pandas numpy matplotlib seaborn statsmodels prophet
jupyter notebook pharma_sales_forecasting.ipynb
```

## Data

[Pharmaceutical Drug Spending Dataset](https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data) — Kaggle.  
Weekly sales data across 8 ATC drug categories.

## Skills Demonstrated

`Time Series Forecasting` · `ARIMA` · `Facebook Prophet` · `Uncertainty Quantification` · `Supply Chain Analytics` · `Python` · `statsmodels`
"""

with open('README.md', 'w') as f:
    f.write(readme.strip())

print("✅ README.md written to project root.")
print()
print("── Files generated ──────────────────────────────────")
import os
for root, dirs, files in os.walk('outputs'):
    for file in sorted(files):
        path = os.path.join(root, file)
        size = os.path.getsize(path)
        print(f"  {path:55s} {size:>8,} bytes")
print("  README.md")
print()
print("✅ Project complete. Push to GitHub.")
